# Notebook 01 — Setup and Data Preparation

**Project:** Study 2 — Chatbot Neuroticism (DGK Lab)  
**Pipeline stage:** 1 of 8 (NLP Colab notebooks) + 1 integration Rmd  
**Runtime:** ~5 minutes  
**API costs:** $0 (no external API calls in this notebook)

## Purpose

This notebook is the foundation for the entire NLP pipeline. It:

1. Mounts Google Drive and verifies the project structure exists
2. Loads the three input files from the quantitative notebook (`composite_scores.csv`, `analytic_data.csv`, `oe_and_chat.csv`)
3. Parses the `chat_log` JSON column into a long-form turn-level dataframe
4. Reshapes the open-ended responses into long form
5. Validates internal consistency (counts match across files, no empty chat_logs, etc.)
6. Computes basic descriptive statistics for the corpus
7. Writes processed data to `data/derived/` for downstream notebooks

**No analyses are performed here.** This is purely data preparation.

## Outputs written to `Study 2/data/derived/`

- `chat_turns_long.parquet` — one row per turn × 135 participants (row count computed at runtime from the Study 2 corpus)
- `open_ends_long.parquet` — one row per OE response (expected 810 rows = 135 × 6 fields)
- `participant_metadata.parquet` — one row per participant with condition, key composites, conversation-level aggregates, and an `n_primary_oe_blank` flag (0–4)
- `corpus_summary.json` — overall descriptives (turn counts, token counts, OE lengths)
- `oe_fields_reference.json` — OE field metadata (order, is_primary, prompt summary) for downstream notebooks

## Before running

Verify these files exist in `Study 2/data/processed/`:
- `composite_scores.csv`
- `analytic_data.csv`  
- `oe_and_chat.csv`

These are produced by `study2_core_quantitative.Rmd` (the R notebook). If you haven't re-knit that recently, do so first.


## 1. Environment setup

In [ ]:
# Standard imports
import json
import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'numpy:  {np.__version__}')
print(f'Run timestamp: {datetime.now().isoformat(timespec="seconds")}')

## 2. Project paths

Detects whether we're running in Google Colab or locally and sets paths accordingly. **You may need to adjust `PROJECT_ROOT` for your specific Google Drive layout** — the default assumes the path used in your quantitative notebook.

In [ ]:
# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Path matches the structure from the R notebook
    PROJECT_ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
                        'DGK Lab - AI Training Data, CharacterAI, Tort Liability/'
                        'Analysis/Study 2')
else:
    # Local fallback — adjust if running outside Colab
    PROJECT_ROOT = Path.cwd().parent if (Path.cwd().name == 'NLP') else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_DERIVED   = PROJECT_ROOT / 'data' / 'derived'
OUTPUTS_NLP    = PROJECT_ROOT / 'outputs' / 'NLP'

# Create output directories if they don't exist
DATA_DERIVED.mkdir(parents=True, exist_ok=True)
OUTPUTS_NLP.mkdir(parents=True, exist_ok=True)

print(f'Project root:    {PROJECT_ROOT}')
print(f'  exists:        {PROJECT_ROOT.exists()}')
print(f'Processed data:  {DATA_PROCESSED}')
print(f'  exists:        {DATA_PROCESSED.exists()}')
print(f'Derived output:  {DATA_DERIVED}')
print(f'NLP outputs:     {OUTPUTS_NLP}')

assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
assert DATA_PROCESSED.exists(), f'DATA_PROCESSED not found: {DATA_PROCESSED}'

## 3. Load input files

Loads three CSVs produced by the R notebook. We use `analytic_data.csv` as the source of truth for the analytic sample because it carries the exclusion flags. We use `composite_scores.csv` for the moderation analyses later in the pipeline (it has cleaned composite scores). And `oe_and_chat.csv` is our text corpus.

In [ ]:
composite = pd.read_csv(DATA_PROCESSED / 'composite_scores.csv')
analytic  = pd.read_csv(DATA_PROCESSED / 'analytic_data.csv', low_memory=False)
oe_chat   = pd.read_csv(DATA_PROCESSED / 'oe_and_chat.csv')

print(f'composite_scores: {composite.shape[0]} rows × {composite.shape[1]} cols')
print(f'analytic_data:    {analytic.shape[0]} rows × {analytic.shape[1]} cols')
print(f'oe_and_chat:      {oe_chat.shape[0]} rows × {oe_chat.shape[1]} cols')

# Sanity check: all three should agree on the analytic sample
assert len(composite) == len(oe_chat), (
    f'Row count mismatch: composite={len(composite)}, oe_chat={len(oe_chat)}'
)
print(f'\nAll three files have consistent N={len(composite)}.')

### 3.1 Verify analytic-sample composition

The R notebook reported N=135 (HighN=71, LowN=64) for the primary analytic sample after exclusions. This should match exactly.

In [ ]:
print('Condition counts in oe_and_chat.csv:')
print(oe_chat['condition_label'].value_counts())
print(f'\nTotal N: {len(oe_chat)}')

# Internal consistency: confirm condition labels are clean and the split is roughly balanced.
# We no longer hard-code an expected N here — the R notebook is the source of truth on sample
# size. This notebook just reports the actual composition descriptively.
actual_n = len(oe_chat)
actual_high = (oe_chat['condition_label'] == 'HighN').sum()
actual_low  = (oe_chat['condition_label'] == 'LowN').sum()

if actual_high + actual_low != actual_n:
    print(f'\n[WARNING] Condition labels do not account for all participants.')
    print(f'  HighN + LowN = {actual_high + actual_low}, but total N = {actual_n}')
    print(f'  Check for unexpected condition_label values in oe_and_chat.csv.')
else:
    print(f'\n[OK] Condition labels well-formed (HighN={actual_high} + LowN={actual_low} = N={actual_n}).')


## 4. Parse `chat_log` JSON into long-form turn data

The `chat_log` column is a JSON array, one element per turn. We parse it into a long-form dataframe with one row per turn. This is the structure all subsequent text-analysis notebooks will read.

In [ ]:
def parse_chat_log(row):
    """Parse one participant's chat_log into a list of turn dicts."""
    sona_id   = row['sona_id']
    cond      = row['condition_label']
    cb_slot   = row['cb_slot']
    log_raw   = row['chat_log']

    if pd.isna(log_raw) or log_raw == '':
        return []

    try:
        log = json.loads(log_raw)
    except json.JSONDecodeError as e:
        print(f'[ERROR] sona_id={sona_id}: invalid JSON ({e})')
        return []

    parsed = []
    for turn_idx, turn in enumerate(log):
        # Pull out usage info if present (some sessions may have it from OpenRouter)
        usage = turn.get('usage') or {}
        completion_tokens = usage.get('completion_tokens') if isinstance(usage, dict) else None
        prompt_tokens     = usage.get('prompt_tokens')     if isinstance(usage, dict) else None

        content = turn.get('content', '') or ''

        parsed.append({
            'sona_id':            sona_id,
            'condition_label':    cond,
            'cb_slot':            cb_slot,
            'turn_idx':           turn_idx,            # 0-based, includes both user and bot
            'round':              turn.get('round'),   # original round number from data
            'role':               turn.get('role'),    # 'user' or 'assistant'
            'content':            content,
            'n_chars':            len(content),
            'n_words':            len(content.split()) if content else 0,
            'completion_tokens':  completion_tokens,
            'prompt_tokens':      prompt_tokens,
            'ts':                 turn.get('ts'),
            'elapsed_ms':         turn.get('elapsed'),
        })
    return parsed

# Apply across all rows and flatten
all_turns = []
for _, row in oe_chat.iterrows():
    all_turns.extend(parse_chat_log(row))

chat_turns = pd.DataFrame(all_turns)
print(f'Parsed {len(chat_turns):,} total turns from {oe_chat["sona_id"].nunique()} participants.')
print(f'\nRole distribution:')
print(chat_turns['role'].value_counts())
print(f'\nTurns per participant (descriptives):')
print(chat_turns.groupby('sona_id').size().describe().round(1))

### 4.1 Add user-turn and bot-turn indices within each conversation

For per-turn analyses (e.g., user-bot alignment over time), we want indices that count user turns and bot turns separately within each conversation. `turn_idx` interleaves them; `user_turn_idx` and `bot_turn_idx` count within role.

In [ ]:
def add_role_indices(df):
    df = df.copy()
    df['user_turn_idx'] = (
        df.assign(_is_u=(df['role']=='user').astype(int))
          .groupby('sona_id')['_is_u'].cumsum() - 1
    )
    df.loc[df['role']!='user', 'user_turn_idx'] = pd.NA

    df['bot_turn_idx'] = (
        df.assign(_is_b=(df['role']=='assistant').astype(int))
          .groupby('sona_id')['_is_b'].cumsum() - 1
    )
    df.loc[df['role']!='assistant', 'bot_turn_idx'] = pd.NA

    # Convert to nullable integers
    df['user_turn_idx'] = df['user_turn_idx'].astype('Int64')
    df['bot_turn_idx']  = df['bot_turn_idx'].astype('Int64')
    return df

chat_turns = add_role_indices(chat_turns)

# Spot check on one participant
sample_sona = chat_turns['sona_id'].iloc[0]
print(f'Sample turn structure for sona_id={sample_sona}:')
print(chat_turns[chat_turns['sona_id']==sample_sona]
      [['turn_idx','role','user_turn_idx','bot_turn_idx','n_chars','n_words']]
      .head(10).to_string(index=False))

### 4.2 Reshape open-ended responses into long form

Every participant has up to 6 free-text responses (4 primary OEs plus 2 demographic free-texts). We reshape into one row per response × participant. This format makes coding easier downstream.

In [ ]:
# OE_FIELDS metadata: order, primary/secondary flag, and the VERBATIM question text
# from experiment.js (not a paraphrase). The verbatim text is what gets passed to
# downstream LLMs (rubric development in Notebook 03, LLM coding in Notebook 04) so
# the models can interpret responses in the same context the participant had.
#
# For oe_responsibility and oe_compensation, the question references "the scenario
# you read about a user experiencing distress" — that scenario is the corporate
# liability vignette participants read on a prior page (experiment.js line 643).
# We store it separately as `scenario_context` so downstream notebooks can include
# it in the prompt for those two questions.

SCENARIO_LIABILITY = (
    "Imagine that a user with pre-existing anxiety and depression spent several months "
    "having daily conversations with Sylva. The user became emotionally dependent on the "
    "chatbot and experienced significant psychological distress when the service was "
    "temporarily unavailable. The user\u2019s mental health symptoms worsened during this "
    "period."
)

OE_FIELDS = {
    'oe_experience': {
        'order': 1, 'is_primary': True,
        'prompt_summary': 'Describe your overall experience conversing with Sylva. What stood out to you about the interaction?',
        'scenario_context': None,
    },
    'oe_person_or_tool': {
        'order': 2, 'is_primary': True,
        'prompt_summary': 'Did Sylva feel more like a person, a tool, or something else entirely? Please explain your answer.',
        'scenario_context': None,
    },
    'oe_responsibility': {
        'order': 3, 'is_primary': True,
        'prompt_summary': 'In the scenario you read about a user experiencing distress, who do you think bears the most responsibility for what happened, and why?',
        'scenario_context': SCENARIO_LIABILITY,
    },
    'oe_compensation': {
        'order': 4, 'is_primary': True,
        'prompt_summary': 'Should the company that created Sylva compensate the affected user? If so, how much and why? If not, why not?',
        'scenario_context': SCENARIO_LIABILITY,
    },
    'd_final_thoughts': {
        'order': 5, 'is_primary': False,
        'prompt_summary': 'Is there anything else you would like to share with us about your experience in this study or your thoughts about AI and moral responsibility?',
        'scenario_context': None,
    },
    'd_problems': {
        'order': 6, 'is_primary': False,
        'prompt_summary': 'Was any part of the procedure unclear? Did you have any problems completing any of the tasks?',
        'scenario_context': None,
    },
}

rows = []
for _, r in oe_chat.iterrows():
    for field, meta in OE_FIELDS.items():
        text = r.get(field)
        text_str = '' if pd.isna(text) else str(text).strip()
        rows.append({
            'sona_id':         r['sona_id'],
            'condition_label': r['condition_label'],
            'cb_slot':         r['cb_slot'],
            'oe_field':        field,
            'oe_order':        meta['order'],
            'is_primary':      meta['is_primary'],
            'prompt_summary':  meta['prompt_summary'],
            'response':        text_str,
            'is_blank':        text_str == '',
            'n_chars':         len(text_str),
            'n_words':         len(text_str.split()) if text_str else 0,
        })

open_ends = pd.DataFrame(rows)
print(f'Reshaped open-ended responses: {len(open_ends):,} rows '
      f'({oe_chat["sona_id"].nunique()} participants × {len(OE_FIELDS)} fields)')
print()
print('Non-blank counts by field:')
print(open_ends.groupby('oe_field').agg(
    n_total=('response', 'size'),
    n_nonblank=('is_blank', lambda s: (~s).sum()),
    pct_nonblank=('is_blank', lambda s: 100 * (~s).sum() / len(s)),
    mean_chars=('n_chars', 'mean'),
    median_chars=('n_chars', 'median'),
).round(1))


## 5. Build participant-level metadata

One row per participant with condition, key composite scores, and conversation-level summary metrics. This is what the LLM coding notebooks will read to provide context, and what the integration Rmd will join NLP features to.

In [ ]:
# Conversation-level aggregates from the parsed chat data
conv_aggs = (
    chat_turns.groupby('sona_id')
    .agg(
        n_turns_total=('turn_idx', 'count'),
        n_user_turns=('role', lambda s: (s == 'user').sum()),
        n_bot_turns=('role', lambda s: (s == 'assistant').sum()),
        user_total_chars=('n_chars', lambda s: s[chat_turns.loc[s.index, 'role'] == 'user'].sum()),
        bot_total_chars=('n_chars',  lambda s: s[chat_turns.loc[s.index, 'role'] == 'assistant'].sum()),
        user_mean_chars=('n_chars',  lambda s: s[chat_turns.loc[s.index, 'role'] == 'user'].mean()),
        bot_mean_chars=('n_chars',   lambda s: s[chat_turns.loc[s.index, 'role'] == 'assistant'].mean()),
    )
    .reset_index()
)

# Per-participant flag: how many of the four primary OE fields were left blank.
# Used downstream by Notebook 04 (LLM coding) to skip all-blank participants
# and by the integration Rmd to decide whether to include/exclude per analysis.
oe_blank_flag = (
    open_ends[open_ends['is_primary']]
    .groupby('sona_id')['is_blank'].sum()
    .rename('n_primary_oe_blank')
    .reset_index()
)
oe_blank_flag['n_primary_oe_blank'] = oe_blank_flag['n_primary_oe_blank'].astype(int)

# Pull key composites from the R-produced file
# Note: Study 2's focal single-item check, mc_neuroticism ("How anxious and worried
# did Sylva seem?"), is unipolar/scorable and IS pulled here, alongside the
# held-constant diffusion checks mc_agreeableness and mc_extraversion. (Contrast
# Study 1, where the focal single-item was bipolar-phrased and therefore dropped.)
key_cols = [
    'sona_id', 'condition_label', 'cb_slot',
    'tipi_self_agree', 'tipi_self_extra', 'tipi_self_consc',
    'tipi_self_emostab', 'tipi_self_open',
    'tipi_bot_agree', 'tipi_bot_extra', 'tipi_bot_emostab',
    'mc_neuroticism', 'mc_agreeableness', 'mc_extraversion',
    'psr_total', 'psr_connection', 'psr_desire', 'psr_caring',
    'ec_total', 'trust_total', 'et_total',
    'ma_total', 'mp_total',
    'liab_corp', 'liab_chat', 'liab_user', 'liab_harm',
    'phq4_total', 'gaais_pos', 'gaais_neg',
    'd_age', 'd_chatbot_experience', 'free_speech_avg',
    'legal_expertise',
]
available = [c for c in key_cols if c in composite.columns]
missing   = [c for c in key_cols if c not in composite.columns]
if missing:
    print(f'[NOTE] Composite columns not found (will be skipped): {missing}')

participant_meta = (
    composite[available]
    .merge(conv_aggs,    on='sona_id', how='left')
    .merge(oe_blank_flag, on='sona_id', how='left')
)
# If any participant has no OE rows (shouldn't happen given the reshape), fill with 0
participant_meta['n_primary_oe_blank'] = participant_meta['n_primary_oe_blank'].fillna(0).astype(int)

print(f'Participant metadata: {participant_meta.shape[0]} rows × {participant_meta.shape[1]} cols')
print()
print(f'OE-blank distribution (0=wrote all 4 primary OEs, 4=blank on all):')
print(participant_meta['n_primary_oe_blank'].value_counts().sort_index())
print()
print('Sample (first 3 participants):')
print(participant_meta.head(3).to_string(index=False))


## 6. Validation checks

Hard checks that the data is internally consistent. If any of these fail, do not proceed to Notebook 02.

In [ ]:
checks = []

# Check 1: every participant has at least one user turn AND one bot turn
min_per_role = chat_turns.groupby(['sona_id','role']).size().unstack(fill_value=0)
no_user = (min_per_role.get('user', pd.Series(dtype=int)) == 0).sum()
no_bot  = (min_per_role.get('assistant', pd.Series(dtype=int)) == 0).sum()
checks.append(('All participants have user turns', no_user == 0, f'{no_user} participants have 0 user turns', 'FAIL'))
checks.append(('All participants have bot turns',  no_bot == 0,  f'{no_bot} participants have 0 bot turns',  'FAIL'))

# Check 2: no orphan participants in any of the three reshape outputs
n_meta = participant_meta['sona_id'].nunique()
n_chat = chat_turns['sona_id'].nunique()
n_oe   = open_ends['sona_id'].nunique()
checks.append(('Same N across files', n_meta == n_chat == n_oe, f'meta={n_meta}, chat={n_chat}, oe={n_oe}', 'FAIL'))

# Check 3: no duplicate sona_ids in metadata
dup_meta = participant_meta['sona_id'].duplicated().sum()
checks.append(('No duplicate sona_ids in metadata', dup_meta == 0, f'{dup_meta} duplicates found', 'FAIL'))

# Check 4: chat_log non-empty for all participants
empty_chats = (chat_turns.groupby('sona_id').size() == 0).sum()
checks.append(('All chat_logs non-empty', empty_chats == 0, f'{empty_chats} empty conversations', 'FAIL'))

# Check 5: condition column is well-formed (only HighN / LowN)
valid_cond = chat_turns['condition_label'].isin(['HighN', 'LowN']).all()
bad_cond_count = (~chat_turns['condition_label'].isin(['HighN', 'LowN'])).sum()
checks.append(('Condition labels valid', valid_cond, f'{bad_cond_count} rows with unexpected labels', 'FAIL'))

# Check 6: open-ends — flag (not fail) participants with all 4 primary responses blank.
# This is a real data state (participant chose not to answer), not a corruption.
# These participants are tagged in participant_metadata.parquet via n_primary_oe_blank
# and downstream notebooks (esp. Notebook 04 LLM coding) decide how to handle them.
primary_blank_counts = (
    open_ends[open_ends['is_primary']]
    .groupby('sona_id')['is_blank'].sum()
)
all_blank = (primary_blank_counts == 4).sum()
checks.append(('No participants with all 4 primary OEs blank', all_blank == 0,
               f'{all_blank} participants have all 4 primary OEs blank (tagged in participant_metadata)',
               'WARN'))

# Print validation report
print('Validation checks:')
print('-' * 70)
any_failures = False
any_warnings = False
for name, passed, detail, severity in checks:
    if passed:
        marker = '[OK]    '
    elif severity == 'WARN':
        marker = '[WARN]  '
        any_warnings = True
    else:
        marker = '[FAIL]  '
        any_failures = True
    print(f'  {marker}{name}')
    if not passed:
        print(f'         -> {detail}')
print('-' * 70)
if any_failures:
    print('VALIDATION FAILED. Review FAIL items above before proceeding to Notebook 02.')
elif any_warnings:
    print('All hard checks passed. WARN items are non-blocking but worth noting. Safe to proceed.')
else:
    print('All checks passed. Safe to proceed.')


## 7. Corpus descriptives

Top-level numbers for the methods section. These should match the values you'd see in `outputs/R/figures/`.

In [ ]:
# Conversation-level descriptives
user_chars = chat_turns.loc[chat_turns['role']=='user', 'n_chars']
bot_chars  = chat_turns.loc[chat_turns['role']=='assistant', 'n_chars']
user_words = chat_turns.loc[chat_turns['role']=='user', 'n_words']
bot_words  = chat_turns.loc[chat_turns['role']=='assistant', 'n_words']

summary = {
    'analytic_n':          int(participant_meta['sona_id'].nunique()),
    'n_high_n':            int((participant_meta['condition_label']=='HighN').sum()),
    'n_low_n':             int((participant_meta['condition_label']=='LowN').sum()),

    'turns_total':         int(len(chat_turns)),
    'user_turns_total':    int((chat_turns['role']=='user').sum()),
    'bot_turns_total':     int((chat_turns['role']=='assistant').sum()),

    'user_chars_mean':     round(float(user_chars.mean()), 1),
    'user_chars_median':   float(user_chars.median()),
    'user_chars_max':      int(user_chars.max()),
    'bot_chars_mean':      round(float(bot_chars.mean()), 1),
    'bot_chars_median':    float(bot_chars.median()),
    'bot_chars_max':       int(bot_chars.max()),

    'user_words_mean':     round(float(user_words.mean()), 1),
    'bot_words_mean':      round(float(bot_words.mean()), 1),

    'oe_total_responses':       int(len(open_ends)),
    'oe_primary_responses':     int(open_ends['is_primary'].sum()),
    'oe_primary_nonblank':      int(((open_ends['is_primary']) & (~open_ends['is_blank'])).sum()),
    'oe_secondary_nonblank':    int(((~open_ends['is_primary']) & (~open_ends['is_blank'])).sum()),

    'oe_primary_chars_mean':    round(float(open_ends.loc[open_ends['is_primary'] & ~open_ends['is_blank'],
                                                          'n_chars'].mean()), 1),
    'oe_primary_chars_median':  float(open_ends.loc[open_ends['is_primary'] & ~open_ends['is_blank'],
                                                    'n_chars'].median()),

    'generated_at': datetime.now().isoformat(timespec='seconds'),
}

print('Corpus summary:')
print('-' * 70)
for k, v in summary.items():
    print(f'  {k:<30s} {v}')
print('-' * 70)

# Per-condition split for sanity
print('\nTurn descriptives by condition:')
by_cond = chat_turns.groupby(['condition_label','role']).agg(
    n_turns=('n_chars', 'count'),
    chars_mean=('n_chars', 'mean'),
    chars_median=('n_chars', 'median'),
).round(1)
print(by_cond)

## 8. Quick visual check

Distribution of user-turn lengths by condition. A sanity check more than an analysis — at this stage we just want to see that the data looks sensible.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# User-turn char distribution by condition
for cond, color in [('LowN', '#B45F06'), ('HighN', '#1E5BA8')]:
    sub = chat_turns[(chat_turns['role']=='user') & (chat_turns['condition_label']==cond)]
    axes[0].hist(sub['n_chars'], bins=40, alpha=0.5, label=cond, color=color)
axes[0].set_xlabel('User turn length (chars)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('User turn length by condition')
axes[0].legend()
axes[0].set_xlim(0, 400)

# Bot-turn char distribution by condition
for cond, color in [('LowN', '#B45F06'), ('HighN', '#1E5BA8')]:
    sub = chat_turns[(chat_turns['role']=='assistant') & (chat_turns['condition_label']==cond)]
    axes[1].hist(sub['n_chars'], bins=40, alpha=0.5, label=cond, color=color)
axes[1].set_xlabel('Bot turn length (chars)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Bot turn length by condition')
axes[1].legend()
axes[1].set_xlim(0, 1200)

plt.tight_layout()
plt.savefig(OUTPUTS_NLP / 'fig01_turn_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSaved figure: {OUTPUTS_NLP / "fig01_turn_length_distributions.png"}')

## 9. Write outputs

Three parquet files for downstream notebooks plus a JSON summary. We use parquet rather than CSV for two reasons: it preserves dtypes (no more reading integer columns as float), and it's much faster to read for the larger dataframes.

In [ ]:
# Ensure pyarrow is available
try:
    import pyarrow
except ImportError:
    print('Installing pyarrow for parquet support...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
    import pyarrow

out_files = {
    DATA_DERIVED / 'chat_turns_long.parquet':       chat_turns,
    DATA_DERIVED / 'open_ends_long.parquet':        open_ends,
    DATA_DERIVED / 'participant_metadata.parquet':  participant_meta,
}

for path, df in out_files.items():
    df.to_parquet(path, index=False)
    size_kb = path.stat().st_size / 1024
    print(f'Wrote {path.name}: {df.shape[0]:>5} rows × {df.shape[1]:>3} cols ({size_kb:>6.1f} KB)')

# Summary JSON
summary_path = DATA_DERIVED / 'corpus_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Wrote {summary_path.name}: corpus_summary.json')

# Also write the OE_FIELDS reference (downstream notebooks need this metadata)
fields_path = DATA_DERIVED / 'oe_fields_reference.json'
with open(fields_path, 'w') as f:
    json.dump(OE_FIELDS, f, indent=2)
print(f'Wrote {fields_path.name}: oe_fields_reference.json')

## 10. Wrap-up checklist

Before moving to Notebook 02, confirm:

- [ ] All validation checks in Section 6 passed (`[OK]` markers)
- [ ] N matches the R notebook (HighN=39, LowN=41 expected)
- [ ] Three parquet files written to `data/derived/`
- [ ] `fig01_turn_length_distributions.png` looks reasonable (no participants with zero-length turns, distributions plausible)
- [ ] `corpus_summary.json` opens and shows expected counts

If all of the above checks out, Notebook 02 (embeddings) can run. If anything looks off, do not proceed — let me know what you see and we'll fix it before building further.

In [ ]:
# Final summary printout
print('=' * 70)
print('NOTEBOOK 01 COMPLETE')
print('=' * 70)
print(f'Analytic N:           {summary["analytic_n"]} (HighN={summary["n_high_n"]}, LowN={summary["n_low_n"]})')
print(f'Conversation turns:   {summary["turns_total"]:,}')
print(f'  User turns:         {summary["user_turns_total"]:,}')
print(f'  Bot turns:          {summary["bot_turns_total"]:,}')
print(f'OE responses:         {summary["oe_primary_nonblank"]} primary non-blank, '
      f'{summary["oe_secondary_nonblank"]} secondary non-blank')
print()
print('Outputs ready for Notebook 02 (embeddings):')
print(f'  {DATA_DERIVED / "chat_turns_long.parquet"}')
print(f'  {DATA_DERIVED / "open_ends_long.parquet"}')
print(f'  {DATA_DERIVED / "participant_metadata.parquet"}')
print(f'  {DATA_DERIVED / "corpus_summary.json"}')
print(f'  {DATA_DERIVED / "oe_fields_reference.json"}')